# Loan Risk Prediction

This notebook builds a reusable function that takes borrower and loan details, predicts Default, returns the probability of default, and assigns a risk level based on probability thresholds.

The preprocessing and model setup follow the earlier notebooks and the selected model is the strongest classifier based on the imbalance-aware evaluation metrics from the training notebook.

In [2]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
df = pd.read_csv('../outputs/cleaned_loan_default.csv')
target_col = 'Default'
id_col = 'LoanID'

numeric_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']
binary_features = ['HasMortgage', 'HasDependents', 'HasCoSigner']
categorical_features = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']
selected_features = numeric_features + binary_features + categorical_features

model_path = Path('../outputs/credit_risk_model.pkl')
with model_path.open('rb') as model_file:
    model_pipeline = pickle.load(model_file)
selected_model_name = getattr(model_pipeline, 'selected_model_name', type(model_pipeline.named_steps['model']).__name__)
print(f'Loaded selected model pipeline: {selected_model_name}')


In [ ]:
def classify_risk(probability):
    if probability < 0.10:
        return 'Low'
    elif probability < 0.30:
        return 'Medium'
    else:
        return 'High'


def predict_loan_risk(borrower_details):
    """Return default prediction, probability, and risk level for one or more borrowers."""
    if isinstance(borrower_details, dict):
        borrower_df = pd.DataFrame([borrower_details])
        single_record = True
    elif isinstance(borrower_details, list):
        borrower_df = pd.DataFrame(borrower_details)
        single_record = False
    else:
        raise TypeError('borrower_details must be a dict or a list of dicts')

    missing = [col for col in selected_features if col not in borrower_df.columns]
    if missing:
        raise ValueError(f'Missing required features: {missing}')

    borrower_df = borrower_df[selected_features].copy()
    probabilities = model_pipeline.predict_proba(borrower_df)[:, 1]
    predictions = model_pipeline.predict(borrower_df)
    output = [{
        'Predicted Default': int(pred),
        'Default Probability': float(prob),
        'Risk Level': classify_risk(float(prob)),
        'Model': selected_model_name,
    } for pred, prob in zip(predictions, probabilities)]
    return output[0] if single_record else output


In [ ]:
sample_inputs = [
    {
        'Age': 34, 'Income': 78000, 'LoanAmount': 40000, 'CreditScore': 710,
        'MonthsEmployed': 18, 'NumCreditLines': 2, 'InterestRate': 8.5, 'LoanTerm': 36,
        'DTIRatio': 0.25, 'Education': "Bachelor's", 'EmploymentType': 'Full-time',
        'MaritalStatus': 'Single', 'HasMortgage': 'No', 'HasDependents': 'No',
        'LoanPurpose': 'Auto', 'HasCoSigner': 'No'
    },
    {
        'Age': 52, 'Income': 65000, 'LoanAmount': 120000, 'CreditScore': 480,
        'MonthsEmployed': 8, 'NumCreditLines': 5, 'InterestRate': 18.2, 'LoanTerm': 60,
        'DTIRatio': 0.67, 'Education': 'High School', 'EmploymentType': 'Unemployed',
        'MaritalStatus': 'Married', 'HasMortgage': 'Yes', 'HasDependents': 'Yes',
        'LoanPurpose': 'Home', 'HasCoSigner': 'No'
    },
    {
        'Age': 41, 'Income': 91000, 'LoanAmount': 72000, 'CreditScore': 610,
        'MonthsEmployed': 24, 'NumCreditLines': 3, 'InterestRate': 11.4, 'LoanTerm': 48,
        'DTIRatio': 0.40, 'Education': "Master's", 'EmploymentType': 'Full-time',
        'MaritalStatus': 'Married', 'HasMortgage': 'No', 'HasDependents': 'Yes',
        'LoanPurpose': 'Business', 'HasCoSigner': 'Yes'
    }
]

predictions = [predict_loan_risk(record) for record in sample_inputs]
pd.DataFrame(predictions)

## Risk level thresholds
- Low: probability < 0.10
- Medium: 0.10 <= probability < 0.30
- High: probability >= 0.30

These thresholds are based on the default rate in the cleaned data and are intended to provide a practical classification for risk review.